# Data Wrangling — Tiki E-commerce (2 bộ)

Nguồn: [Vietnamese Tiki E-commerce Dataset (Kaggle)](https://www.kaggle.com/datasets/michaelminhpham/vietnamese-tiki-e-commerce-dataset)

1. `vietnamese_tiki_products_men_shoes.csv` — giày nam  
2. `vietnamese_tiki_products_women_shoes.csv` — giày nữ  

Mục tiêu gốc của bộ dữ liệu: dự đoán `quantity_sold`.

Mỗi bước có **1 dòng mẫu**. Dòng `# TODO` em làm tương tự (đổi tên cột / file).

## Download data and explore

Luôn bắt đầu bằng **nhìn dữ liệu**, đừng nhảy vào `fillna`. Hỏi: bao nhiêu dòng? cột nào object? giá trị lạ (`...`, tab, rỗng)?

| Cột | Ý nghĩa |
|---|---|
| `id` | Mã sản phẩm Tiki |
| `name` | Tên sản phẩm |
| `description` | Mô tả (có thể rỗng / `...`) |
| `original_price` | Giá gốc (VND) |
| `price` | Giá hiện tại (VND) |
| `fulfillment_type` | Hình thức giao: `dropship`, `tiki_delivery`, `seller_delivery` |
| `brand` | Thương hiệu (nhiều `OEM`, đôi khi dính tab `\\tOEM`) |
| `review_count` | Số đánh giá |
| `rating_average` | Điểm trung bình (0–5; 0 thường = chưa có đánh giá) |
| `favourite_count` | Số lượt yêu thích |
| `pay_later` | Có trả sau hay không |
| `current_seller` | Tên shop |
| `date_created` | Số ngày từ lần cập nhật |
| `number_of_images` | Số ảnh |
| `vnd_cashback` | Số tiền hoàn (VND) |
| `has_video` | Có video hay không |
| `category` | Danh mục (`Root` = chưa gán danh mục rõ) |
| `quantity_sold` | Tổng số đã bán (biến mục tiêu) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

path_men = "tiki_data/vietnamese_tiki_products_men_shoes.csv"
path_women = "tiki_data/vietnamese_tiki_products_women_shoes.csv"

df_men = pd.read_csv(path_men)  # mẫu: đọc file nam
# TODO: đọc file nữ vào df_women (giống dòng trên, đổi path_women)
df_women = None

print("Men shoes:", df_men.shape)  # mẫu: in số dòng, số cột
# TODO: in shape của df_women

df_men.head()  # mẫu: 5 dòng đầu
# TODO: xem 5 dòng đầu df_women

---
# Bộ 1 — Giày nam

## 1. Xử lý giá trị thiếu (Missing Values)

- Chuỗi rỗng / `...` / `"nan"` → `NaN`.
- `fulfillment_type`, `brand`: cột phân loại → **mode**.
- `description`: điền `""` (không xoá dòng).
- `quantity_sold`: biến mục tiêu → `.dropna(subset=["quantity_sold"])`.

In [ ]:
df = df_men.copy()  # mẫu: làm việc trên bản sao

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])  # mẫu: xoá cột index thừa

# TODO: xoá trùng theo id rồi reset_index — gợi ý: drop_duplicates(subset=["id"])
df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].astype(str).str.strip()  # mẫu: cắt khoảng trắng 2 đầu
    # TODO: trong vòng for, replace {"nan": np.nan, "": np.nan, "...": np.nan} cho df[c]
df[c] = df[c].replace({"nan": np.nan, "": np.nan, "...": np.nan})
df["brand"] = df["brand"].str.replace("\t", "", regex=False)  # mẫu: xoá tab trong brand

print(df.isnull().sum())  # mẫu: đếm missing trước xử lý

df["fulfillment_type"] = df["fulfillment_type"].fillna(
    df["fulfillment_type"].mode(dropna=True)[0]
)  # mẫu: cột chữ → điền mode
# TODO: fillna mode cho cột brand (copy mẫu, đổi tên cột)
df["brand"] = df["brand"].fillna(df["brand"].mode(dropna=True)[0])
# TODO: df["description"] fillna bằng ""
df["description"] = df["description"].fillna("")
# TODO: dropna subset=["quantity_sold"] rồi reset_index
df = df.dropna(subset=["quantity_sold"]).reset_index(drop=True)
# TODO: in lại isnull().sum() sau khi xử lý
print(df.isnull().sum())

## 2. Sửa định dạng dữ liệu (Correct Data Format)

In [ ]:
print(df.dtypes)  # mẫu: xem kiểu hiện tại

df["price"] = df["price"].astype(int)  # mẫu: ép price về int
# TODO: ép original_price, quantity_sold, review_count về int (giống dòng trên)
df["original_price"] = df["original_price"].astype(int)
df["quantity_sold"] = df["quantity_sold"].astype(int)
df["review_count"] = df["review_count"].astype(int)

# TODO: ép rating_average về float
df["rating_average"] = df["rating_average"].astype(float)
# TODO: print dtypes các cột vừa ép
print(df[["price", "original_price", "quantity_sold", "review_count", "rating_average"]].dtypes)

## 3. Chuẩn hoá dữ liệu (Data Standardization)

Giá đang là **VND**. Đổi sang **nghìn đồng** cho dễ đọc, và tính **% giảm giá** so với giá gốc.

In [ ]:
df["price_nghin"] = df["price"] / 1000  # mẫu: VND → nghìn đồng
# TODO: cột original_price_nghin = original_price / 1000
df["original_price_nghin"] = df["original_price"] / 1000
# TODO: cột discount_pct = (original_price - price) / original_price * 100
#       dùng np.where(original_price > 0, công_thức, 0)
df["discount_pct"] = np.where(
    df["original_price"] > 0,
    (df["original_price"] - df["price"]) / df["original_price"] * 100,
    0
)
df[["price", "price_nghin"]].head()  # mẫu: xem kết quả
# TODO: head thêm original_price và discount_pct
df[["price", "price_nghin", "original_price", "original_price_nghin", "discount_pct"]].head()

## 4. Chuẩn hoá phạm vi giá trị (Data Normalization)

`price` và `quantity_sold` lệch thang đo rất mạnh → `x / x.max()` về 0–1.

In [ ]:
df["price_normalized"] = df["price"] / df["price"].max()  # mẫu: chia max → 0–1
# TODO: quantity_sold_normalized = quantity_sold / max
df["quantity_sold_normalized"] = df["quantity_sold"] / df["quantity_sold"].max()

# TODO: review_count_normalized = review_count / max
df["review_count_normalized"] = df["review_count"] / df["review_count"].max()
df[["price", "price_normalized"]].head()  # mẫu
# TODO: head thêm quantity_sold và cột normalize tương ứng
df[["price", "price_normalized", "quantity_sold", "quantity_sold_normalized", "review_count", "review_count_normalized"]].head()

## 5. Phân nhóm (Binning)

Chia giá thành Low / Medium / High theo ngưỡng VND thực tế (không dùng min–max đều vì giá max rất lớn, hầu hết sản phẩm sẽ rơi vào Low).

In [ ]:
price_bins = [0, 100_000, 500_000, df["price"].max() + 1]  # mẫu: 3 khoảng giá

df["price_binned"] = pd.cut(
    df["price"], bins=price_bins, labels=["Low", "Medium", "High"], include_lowest=True
)  # mẫu: gán nhãn Low / Medium / High

print(df["price_binned"].value_counts())  # mẫu: đếm mỗi nhóm

# TODO: vẽ bar — gợi ý: df["price_binned"].value_counts().sort_index().plot(kind="bar")
df["price_binned"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Price category")
plt.ylabel("Count")
plt.title("Phân bố giá - Giày nam")
plt.show()
# TODO: plt.xlabel / ylabel / title rồi plt.show()

df[["price", "price_binned"]].head()

## 6. Biến chỉ thị / Biến giả (Dummy Variable)

`fulfillment_type` (chữ) → mỗi hình thức giao một cột 0/1 bằng `pd.get_dummies()`, rồi xoá cột gốc.

In [ ]:
dummy_ff = pd.get_dummies(df["fulfillment_type"], prefix="fulfillment", dtype=int)  # mẫu: cột 0/1

# TODO: gộp dummy vào df — pd.concat([df, dummy_ff], axis=1)
df = pd.concat([df, dummy_ff], axis=1)

# TODO: xoá cột gốc — df.drop(columns=["fulfillment_type"])
df = df.drop(columns=["fulfillment_type"])
df.head()  # mẫu: kiểm tra bảng

## Check and save

Trước khi `to_csv`:

* `isnull().sum()` còn 0 (hoặc đúng chỗ cố ý để thiếu)
* `price`, `quantity_sold` là số
* có cột mới: `price_nghin`, `*_normalized`, `price_binned`, `fulfillment_*`

`index=False` để file không thêm cột index 0,1,2,…


In [ ]:
df.to_csv("tiki_men_shoes_clean.csv", index=False)  # mẫu: lưu file nam
print(df.shape)

# TODO: gán df_men_clean = df để giữ bản sạch bộ nam
df_men_clean = df.copy()
print("Men shoes clean shape:", df_men_clean.shape)

---
---
# Bộ 2 — Giày nữ

Làm **giống bộ nam**, đổi `df_men` → `df_women`. Copy mẫu ở trên, sửa tên biến. Kết thúc bằng **Check and save** (`to_csv`, `index=False`).


In [ ]:
df = df_women.copy()  # mẫu: bắt đầu từ file nữ

# TODO: drop Unnamed: 0 (if có), drop_duplicates id, reset_index — copy bộ nam
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])
# TODO: strip + replace nan/""/"..." trên cột object; xoá \t ở brand
df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].astype(str).str.strip()
    df[c] = df[c].replace({"nan": np.nan, "": np.nan, "...": np.nan})

df["brand"] = df["brand"].str.replace("\t", "", regex=False)

print(df.isnull().sum())  # mẫu

# TODO: fillna mode fulfillment_type (đã có mẫu bộ nam) và brand; description → ""
df["fulfillment_type"] = df["fulfillment_type"].fillna(
    df["fulfillment_type"].mode(dropna=True)[0]
)
df["brand"] = df["brand"].fillna(df["brand"].mode(dropna=True)[0])
df["description"] = df["description"].fillna("")

# TODO: dropna quantity_sold, reset_index, in isnull().sum() lần nữa
df = df.dropna(subset=["quantity_sold"]).reset_index(drop=True)

In [ ]:
df["price"] = df["price"].astype(int)  # mẫu bước 2
# TODO: ép các cột số còn lại (như bộ nam)
df["original_price"] = df["original_price"].astype(int)
df["quantity_sold"] = df["quantity_sold"].astype(int)
df["review_count"] = df["review_count"].astype(int)
df["rating_average"] = df["rating_average"].astype(float)


df["price_nghin"] = df["price"] / 1000  # mẫu bước 3
# TODO: original_price_nghin và discount_pct
df["price_nghin"] = df["price"] / 1000
df["original_price_nghin"] = df["original_price"] / 1000
df["discount_pct"] = np.where(
    df["original_price"] > 0,
    (df["original_price"] - df["price"]) / df["original_price"] * 100,
    0
)
df["price_normalized"] = df["price"] / df["price"].max()  # mẫu bước 4

# TODO: normalize quantity_sold và review_count
df["price_normalized"] = df["price"] / df["price"].max()
df["quantity_sold_normalized"] = df["quantity_sold"] / df["quantity_sold"].max()
df["review_count_normalized"] = df["review_count"] / df["review_count"].max()

# TODO: bước 5 — pd.cut + value_counts + plot (copy mẫu bộ nam, đổi title Women)
price_bins = [0, 100_000, 500_000, df["price"].max() + 1]
df["price_binned"] = pd.cut(
    df["price"], bins=price_bins, labels=["Low", "Medium", "High"], include_lowest=True
)
df["price_binned"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Price category")
plt.ylabel("Count")
plt.title("Phân bố giá - Giày nữ")
plt.show()


dummy_ff = pd.get_dummies(df["fulfillment_type"], prefix="fulfillment", dtype=int)  # mẫu bước 6
# TODO: concat + drop fulfillment_type
dummy_ff = pd.get_dummies(df["fulfillment_type"], prefix="fulfillment", dtype=int)
df = pd.concat([df, dummy_ff], axis=1)
df = df.drop(columns=["fulfillment_type"])
# TODO: to_csv tiki_women_shoes_clean.csv; print shape; df.head()
df_women_clean = df.copy()

print("Women shoes clean shape:", df_women_clean.shape)
df_women_clean.head()